## Data cleaning and Preprocessing

In [ ]:
# adding necessary libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [ ]:
# opening the dataset
data_filepath = "train_data/group_01_train.csv"
data = pd.read_csv(data_filepath)
data

In [ ]:
data.info()

In [ ]:
# here we look at what percentage of the columns are null
missing_values = data.isna().sum().sort_values(ascending=False)
missing_percentage = missing_values[missing_values!=0]/len(data)*100
missing_percentage

In [ ]:
# dropping columns with missing percentage of more than 25%
over_miss_cols = missing_percentage[missing_percentage > 25].index
data = data.drop(columns=over_miss_cols)
print("Dropped columns:")
print(list(over_miss_cols))

In [ ]:
pd.DataFrame({
    'Dtype': data.dtypes,
    'Missing Count': data.isna().sum()
}).query('`Missing Count` > 0').sort_values('Missing Count', ascending=False)

In [ ]:
# Drop rows only if they have missing values in the object columns
object_cols = data.select_dtypes(include='object').columns
before = len(data)
data = data.dropna(subset=object_cols)
after = len(data)

print("Rows removed:", before - after)

In [ ]:
data.info()

In [ ]:
data.nunique()

In [ ]:
# dropping f11 because the only value used in this column is is 'US'
data = data.drop(columns=['f11'])

In [ ]:
# dropping f7 because converting it to a numerical value would need LLMs and NLPs which are not discussed in this course
data = data.drop(columns=['f7'])

In [ ]:
# dropping f8 because of the same reason for the above cell
data = data.drop(columns=['f8'])

In [ ]:
data.nunique()

In [ ]:
# we draw boxplot of dutation between f2 and f13 (f2 - f13) and see if there is any
# meaningful relation between y and this feature.

data['f13'] = pd.to_datetime(data['f13'], format='mixed')
data['f2'] = pd.to_datetime(data['f2'], format='mixed')

data['duration_min'] = (data['f2'] - data['f13']).dt.total_seconds() / 60.0

df_clean = data[data['duration_min'] > 0].copy()

sns.boxplot(data=df_clean, x='y', y='duration_min', showfliers=False)
plt.title('Event Duration vs Target Label (y)')
plt.xlabel('Target Label (y)')
plt.ylabel('Duration (Minutes)')
plt.show()

In [ ]:
# we use f13 to extract hour, day of the week, and season of the record and save them in different columns.

data['f13'] = pd.to_datetime(data['f13'], format='mixed')

def get_season(month):
    if month in [12, 1, 2]:
        return 4
    elif month in [3, 4, 5]:
        return 1
    elif month in [6, 7, 8]:
        return 2
    else:
        return 3
    
data['hour'] = data['f13'].dt.hour
data['day_of_week'] = data['f13'].dt.dayofweek
data['season'] = data['f13'].dt.month.apply(get_season)

In [ ]:
# we show the proportion of each target within each season and see if there is any
# meaningful relation between them.
pd.crosstab(data['season'], data['y'], normalize='index')

In [ ]:
# we draw a histogram to see relationship between hour, and 4 columns which are Day/Night (f31, f32, f33, f34)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
cols = ['f31', 'f32', 'f33', 'f34']
axes_flat = axes.flatten()

for i, col in enumerate(cols):
    sns.histplot(
        data=data, 
        x='hour', 
        hue=col, 
        multiple='dodge', 
        bins=24, 
        shrink=0.8, 
        ax=axes_flat[i]
    )
    axes_flat[i].set_title(f'Relationship Between Extracted Hour and Column {col}')
    axes_flat[i].set_xlabel('Hour of the Day (0-23)')
    axes_flat[i].set_ylabel('Accident Count')
    axes_flat[i].set_xticks(range(0, 24))


plt.tight_layout()
fig.savefig('hour_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# we drop these 4 columns becuase they show high colinearity with `hour`
data = data.drop(columns=['f31', 'f32', 'f33', 'f34'])

In [ ]:
# we drop these f2 and f13 cause their information is extracted and they're not needed anymore.
data = data.drop(columns=['f2', 'f13'])
data.columns

In [ ]:
# splitting the dataset into 15% test, 15% validation and 70% training sets

X = data.drop(columns=['y'])
y = data['y']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=123, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=123, stratify=y_temp
)

print(f"Training set:   {X_train.shape}, Labels: {y_train.shape}")
print(f"Validation set: {X_val.shape}, Labels: {y_val.shape}")
print(f"Test set:       {X_test.shape}, Labels: {y_test.shape}")

In [ ]:
# mean imputer for numerical numbers on the trainig dataset

numeric_cols = X_train.select_dtypes(include='number').columns
imputer = SimpleImputer(strategy='mean')
imputer.fit(X_train[numeric_cols])

X_train.loc[:, numeric_cols] = imputer.transform(X_train[numeric_cols])
X_val.loc[:, numeric_cols] = imputer.transform(X_val[numeric_cols])
X_test.loc[:, numeric_cols] = imputer.transform(X_test[numeric_cols])

In [ ]:
# frequency encoding for text dtype high cardinality columns
cols = ['f9', 'f10', 'f18', 'f21']

for col in cols:
    freq = X_train[col].value_counts()
    X_train.loc[:, col] = X_train[col].map(freq).astype(int)
    X_val.loc[:, col] = X_val[col].map(freq).fillna(0).astype(int)
    X_test.loc[:, col] = X_test[col].map(freq).fillna(0).astype(int)

In [ ]:
# ont-hot-oncoding for text dtype categorical columns
cols = ['f1', 'f12']
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=int)

train_encoded = ohe.fit_transform(X_train[cols])
val_encoded = ohe.transform(X_val[cols])
test_encoded = ohe.transform(X_test[cols])

feature_names = ohe.get_feature_names_out(cols)
train_encoded_df = pd.DataFrame(train_encoded, columns=feature_names, index=X_train.index)
val_encoded_df = pd.DataFrame(val_encoded, columns=feature_names, index=X_val.index)
test_encoded_df = pd.DataFrame(test_encoded, columns=feature_names, index=X_test.index)

X_train = pd.concat([X_train.drop(columns=cols), train_encoded_df], axis=1)
X_val = pd.concat([X_val.drop(columns=cols), val_encoded_df], axis=1)
X_test = pd.concat([X_test.drop(columns=cols), test_encoded_df], axis=1)

In [ ]:
# Converting boolean columns to integers for ML compatibility
bool_cols = X_train.select_dtypes(include='bool').columns

X_train[bool_cols] = X_train[bool_cols].astype(int)
X_val[bool_cols] = X_val[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

In [ ]:
# columns standardization
num_cols = X_train.select_dtypes(include='number').columns

X_train[num_cols] = X_train[num_cols].astype(float)
X_val[num_cols] = X_val[num_cols].astype(float)
X_test[num_cols] = X_test[num_cols].astype(float)

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols]).round(2)
X_val[num_cols] = scaler.transform(X_val[num_cols]).round(2)
X_test[num_cols] = scaler.transform(X_test[num_cols]).round(2)

In [ ]:
# 1. Recombining features
train_final = pd.concat([X_train, y_train], axis=1)
val_final = pd.concat([X_val, y_val], axis=1)
test_final = pd.concat([X_test, y_test], axis=1)

In [ ]:
# dropping duplicates in the training set
print("Training duplicates before:", train_final.duplicated().sum())
train_final = train_final.drop_duplicates()
print("Training duplicates after:", train_final.duplicated().sum())

In [ ]:
# exporting final datasets.
print("\n--- Final Dataset Shapes ---")
print(f"Training Data:   {train_final.shape}")
print(f"Validation Data: {val_final.shape}")
print(f"Test Data:       {test_final.shape}")

train_final.to_csv('train_data/train_preprocessed.csv', index=False)
val_final.to_csv('train_data/val_preprocessed.csv', index=False)
test_final.to_csv('train_data/test_preprocessed.csv', index=False)